# 14 - Train Offline AWR

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but trains **Advantage-Weighted Regression** (Peng et al., 2019) with two heads:

- **Q head** (`action_value`) regresses onto in-batch Monte Carlo returns `G = r + γ G_next` (`γ` from `gamma_step` / `gamma_episode_*` / `gamma_task_*`, same done-code table as DQN; no TD(λ), no value bootstrap).
- **Policy head** (`action`) is weighted log-likelihood, `w = min(exp(A / β), ω_max)` with `A = G − max_a Q(s, a)`.

Act with the policy: `action_head="action"`. `get_action(..., temperature=0)` is greedy-π.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import AwrObjective
from mouse_core.models import LoRAConfig, Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead, DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"                   # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-awr-offline"           # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                                        # number of discrete actions predicted by each head
MAX_OBS_DISCRETE = 64                                  # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                         # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
ADVANTAGE_TEMPERATURE = 0.05                 # AWR β (Peng et al., 2019, all reported Gym tasks)
WEIGHT_CLIP = 20.0                           # AWR ω_max (Peng et al., 2019)
POLICY_COEF = 1.0                            # weight on L_π (not in AWR; they train Q and π separately)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)


## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` → `predictions["action_value"]` and `DiscreteActionHead` → `predictions["action"]`. `get_action` reads the policy (`action_head="action"`).

The backbone exposes `hidden_dim`, and the embedder and heads use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so heads are read from it).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(train_kernel="flex", decode_kernel="flex", dtype=preferred_dtype(device), pretrained="Qwen/Qwen3-0.6B", lora=LoRAConfig(rank=16, alpha=32))

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head_kwargs = dict(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={
        "action_value": DiscreteActionValueHead(**head_kwargs),
        "action": DiscreteActionHead(**head_kwargs),
    },
    action_head="action",
    reasoner=None,
    recurrence=None,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions)` computes the AWR loss and metrics. `G` is the in-window Monte Carlo return `r + γ G_next`: `gamma_step` discounts running steps, `gamma_episode_*` / `gamma_task_*` discount (or stop, when `0`) at those boundaries (no delayed network, no value bootstrap).
4. `AdamW` updates weights. The backbone base is frozen bf16 and trains through its fp32 LoRA adapters (`lora=LoRAConfig(...)`); encoder and heads are fp32 too, so every update lands in fp32 with no master weights. To fine-tune the whole backbone instead, omit `lora=` and build it with `dtype=torch.float32`.

`AwrObjective` fits Q to the in-run Monte Carlo return and the policy to actions weighted by clipped `exp(A / β)`. `advantage_temperature`, `weight_clip`, and `policy_coef` have no defaults.


In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
objective = AwrObjective(
    advantage_temperature=ADVANTAGE_TEMPERATURE,
    weight_clip=WEIGHT_CLIP,
    policy_coef=POLICY_COEF,
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)

def run_train(*, model: Model, optimizer: AdamW, objective: AwrObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        loss, metrics = objective(objective_data.to(device), out.predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_loss']:.3f}  pi={metrics['policy_loss']:.3f}  A={metrics['advantage_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
